# 06 — Diagnostics

**One job:** audit failures, uncertainty, label style, train-test behavior, and empty images

Run from top to bottom. Slow actions are off by default.

In [ ]:
%pip install -q ultralytics pycocotools segmentation-models-pytorch

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
import sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_DIR = Path("/content/drive/MyDrive/Google Colab/Solar Filament Segmentation Challenge")
sys.path.insert(0, str(PROJECT_DIR))
%load_ext autoreload
%autoreload 2

from src.config import DEFAULT_PIPELINE_SETTINGS, TRAINING_SETTINGS
from src.data import copy_raw_data_to_local, load_records, make_split, verify_or_write_split

LOCAL_DATA_DIR = copy_raw_data_to_local(PROJECT_DIR / "data/raw")
TRAIN_IMAGES_DIR = LOCAL_DATA_DIR / "train/train_images"
TEST_IMAGES_DIR = LOCAL_DATA_DIR / "test/test_images"
ANNOTATIONS_PATH = LOCAL_DATA_DIR / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"
SPLIT_MANIFEST_PATH = PROJECT_DIR / "data/processed/split_manifest.json"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

In [ ]:
RUN_HUMAN_AGREEMENT = False
RUN_EMPTY_IMAGE_DIAGNOSTIC = False
YOLO_CHECKPOINT = PROJECT_DIR / "models/stage1/yolo26x_filament_detection/weights/best.pt"
UNET_CHECKPOINT = PROJECT_DIR / "models/stage2/crop_unet_b5/best_crop_unet_b5.pt"

## 1. Failure audit

The saved old-pipeline audit found 468 validation FNs: 219 crop-coverage failures (192 with no proposal) and 230 raw-mask failures. Read the migrated `annotation_details.csv`, `traces.jsonl`, and `coverage_headroom.csv` here before changing the architecture.

In [ ]:
audit_files = sorted((PROJECT_DIR / "experiments/diagnostics").rglob("annotation_details.csv"))
if audit_files:
    audit = pd.read_csv(audit_files[-1])
    display(audit.head())
else:
    print("No migrated stage-audit table found.")

## 2. Confidence break-even

In [ ]:
pq, sq = 0.43004963808881497, 0.6825533378063571
break_even = 2 * sq / pq - 1
print(f"Removing one FP is worthwhile while it costs fewer than {break_even:.2f} TPs.")

## 3. Bootstrap and paired comparisons

In [ ]:
from src.metrics import bootstrap_pq, paired_pq_delta, score_prediction_file

print("Load two migrated by_annotator.csv files and use paired_pq_delta(control, candidate).")
print("The saved marginal bootstrap SD is 0.0149, roughly a ±0.03 95% interval.")

## 4. Gap checks: annotator count, aggregation, style, and firing rate

Saved checks: PQ for 1/2/3-annotator images = 0.4398/0.4350/0.4252; pooled/mean-record/mean-image/first-record = 0.4331/0.4144/0.4214/0.4297; final masks per image = 7.28 test vs 7.26 validation. Known annotator style does not explain the leaderboard gap.

In [ ]:
coco, records = load_records(ANNOTATIONS_PATH, TRAIN_IMAGES_DIR)
_, _, validation_records = verify_or_write_split(records, SPLIT_MANIFEST_PATH)
records_per_image = pd.Series([record["physical_id"] for record in validation_records]).value_counts()
display(records_per_image.value_counts().sort_index().rename("validation_images").to_frame())

## 5. Empty test images

The saved C0 submission had four zero-mask images. This cell rechecks only those images at detector confidence 0.01 and pipeline confidence 0.05; it is off by default.

In [ ]:
if RUN_EMPTY_IMAGE_DIAGNOSTIC:
    from PIL import Image
    from ultralytics import YOLO
    from src.inference import predict_instances
    from src.models import load_checkpoint

    statistics_files = sorted((PROJECT_DIR / "submissions").rglob("image_statistics.csv"))
    assert statistics_files, "No submission image_statistics.csv found"
    empty = pd.read_csv(statistics_files[-1]).query("final_instances == 0")
    detector = YOLO(str(YOLO_CHECKPOINT))
    unet, checkpoint = load_checkpoint(UNET_CHECKPOINT, "b5", device)
    low_settings = {**DEFAULT_PIPELINE_SETTINGS, "yolo_confidence": 0.05}
    for row in empty.itertuples():
        image_path = TEST_IMAGES_DIR / row.image
        result = detector.predict(str(image_path), imgsz=1280, conf=0.01, iou=0.20, max_det=300, end2end=False,
                                  device=0 if device.type == "cuda" else "cpu", verbose=False)[0]
        masks, counts = predict_instances(image_path, detector, unet, device, 0.45, low_settings)
        scores = result.boxes.conf.detach().cpu().numpy()
        print(row.image, "boxes >=0.05:", int((scores >= 0.05).sum()), "fallback masks:", len(masks), counts)

## 6. False-positive anatomy

Use the official per-record files and saved masks to inspect full-size blobs, duplicate elongated structures, and unmatched low-confidence proposals. Keep this descriptive; do not tune on hand-picked examples.